<a href="https://colab.research.google.com/github/GinnaGomez09/proyecto_aplicado_javeriana/blob/main/notebooks/02_limpieza_normalizacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# 1. Clonar repositorio y preparar entorno
# ============================================================

!git clone https://github.com/GinnaGomez09/proyecto_aplicado_javeriana.git
%cd proyecto_aplicado_javeriana

Cloning into 'proyecto_aplicado_javeriana'...
remote: Enumerating objects: 265, done.
remote: Counting objects: 100% (123/123), done.
remote: Compressing objects: 100% (104/104), done.
remote: Total 265 (delta 84), reused 18 (delta 18), pack-reused 142 (from 1)
Receiving objects: 100% (265/265), 3.64 MiB | 4.65 MiB/s, done.
Resolving deltas: 100% (138/138), done.
/content/proyecto_aplicado_javeriana


In [2]:
# ============================================================
# 2. Importar librerías
# ============================================================

import pandas as pd
import numpy as np
import re
import unicodedata

In [3]:
# ============================================================
# 3. Cargar datasets
# ============================================================

recetas_path = "data/raw/recetas/recetas_ingredientes.csv"
tcac_path = "data/raw/tcac/tcac.csv"

recetas = pd.read_csv(recetas_path)
tcac = pd.read_csv(tcac_path)

print("Recetas:", recetas.shape)
print("TCAC:", tcac.shape)

recetas.head()

Recetas: (31587, 16)
TCAC: (773, 41)


,receta_uuid,receta_titulo,receta_url,ingrediente_id,ingrediente_linea,cantidad_original,cantidad_conv,cantidad_min,cantidad_max,unidad,ingrediente_nombre,tcac_alimento_codigo,tcac_alimento_nombre,match_method,match_score,cantidad_gramos_est
0,86af61e4-e16a-11ed-9591-a96d6180cd25,Arepas de Queso,https://www.mycolombianrecipes.com/es/arepas-d...,1,1 taza de harina de arepa blanca o amarilla,1,1.000000,NaN,NaN,taza,harina de arepa blanca o amarilla,NaN,NaN,NaN,NaN,NaN
1,86af61e4-e16a-11ed-9591-a96d6180cd25,Arepas de Queso,https://www.mycolombianrecipes.com/es/arepas-d...,2,1 taza de agua tibia,1,1.000000,NaN,NaN,taza,agua tibia,NaN,NaN,NaN,NaN,NaN
2,86af61e4-e16a-11ed-9591-a96d6180cd25,Arepas de Queso,https://www.mycolombianrecipes.com/es/arepas-d...,3,⅓ taza de queso mozzarella o queso blanco rallado,⅓,0.333333,NaN,NaN,taza,queso mozzarella o queso blanco rallado,NaN,NaN,NaN,NaN,NaN
3,86af61e4-e16a-11ed-9591-a96d6180cd25,Arepas de Queso,https://www.mycolombianrecipes.com/es/arepas-d...,4,2 cucharadas de mantequilla,2,2.000000,NaN,NaN,cucharadas,mantequilla,NaN,NaN,NaN,NaN,NaN
4,86af61e4-e16a-11ed-9591-a96d6180cd25,Arepas de Queso,https://www.mycolombianrecipes.com/es/arepas-d...,5,Sal,NaN,NaN,NaN,NaN,NaN,Sal,NaN,NaN,NaN,NaN,NaN


In [4]:
# ============================================================
# 4. Definición de columnas clave
# ============================================================

col_linea = "ingrediente_linea"
col_ingrediente = "ingrediente_nombre"
col_unidad = "unidad"

print("Columnas utilizadas:")
print(col_linea, col_ingrediente, col_unidad)

Columnas utilizadas:
ingrediente_linea ingrediente_nombre unidad


In [5]:
# ============================================================
# 5. Funciones de limpieza de texto
# ============================================================

def quitar_tildes(texto):
    if pd.isna(texto):
        return np.nan

    texto = str(texto)
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join([c for c in texto if not unicodedata.combining(c)])
    return texto


def limpiar_texto(texto):
    if pd.isna(texto):
        return np.nan

    texto = str(texto)

    # minúsculas
    texto = texto.lower()

    # quitar tildes
    texto = quitar_tildes(texto)

    # eliminar caracteres especiales
    texto = re.sub(r"[^a-z0-9\s\/\.,]", " ", texto)

    # normalizar espacios
    texto = re.sub(r"\s+", " ", texto).strip()

    return texto

In [6]:
# ============================================================
# 6. Normalización de unidades
# ============================================================

def normalizar_unidad(texto):
    if pd.isna(texto):
        return np.nan

    texto = limpiar_texto(texto)

    equivalencias = {
        "cda": "cucharada",
        "cdas": "cucharada",
        "cucharadas": "cucharada",

        "cdta": "cucharadita",
        "cdtas": "cucharadita",
        "cucharaditas": "cucharadita",

        "tazas": "taza",
        "tz": "taza",

        "g": "gramo",
        "gr": "gramo",
        "gramos": "gramo",

        "kg": "kilogramo",
        "kilo": "kilogramo",

        "ml": "mililitro",
        "l": "litro",

        "unidades": "unidad"
    }

    return equivalencias.get(texto, texto)

In [7]:
# ============================================================
# 7. Aplicación de limpieza y normalización
# ============================================================

recetas_limpias = recetas.copy()

recetas_limpias["ingrediente_linea_limpia"] = recetas_limpias[col_linea].apply(limpiar_texto)
recetas_limpias["ingrediente_nombre_limpio"] = recetas_limpias[col_ingrediente].apply(limpiar_texto)

if col_unidad in recetas_limpias.columns:
    recetas_limpias["unidad_limpia"] = recetas_limpias[col_unidad].apply(normalizar_unidad)

recetas_limpias.head()

,receta_uuid,receta_titulo,receta_url,ingrediente_id,ingrediente_linea,cantidad_original,cantidad_conv,cantidad_min,cantidad_max,unidad,ingrediente_nombre,tcac_alimento_codigo,tcac_alimento_nombre,match_method,match_score,cantidad_gramos_est,ingrediente_linea_limpia,ingrediente_nombre_limpio,unidad_limpia
0,86af61e4-e16a-11ed-9591-a96d6180cd25,Arepas de Queso,https://www.mycolombianrecipes.com/es/arepas-d...,1,1 taza de harina de arepa blanca o amarilla,1,1.000000,NaN,NaN,taza,harina de arepa blanca o amarilla,NaN,NaN,NaN,NaN,NaN,1 taza de harina de arepa blanca o amarilla,harina de arepa blanca o amarilla,taza
1,86af61e4-e16a-11ed-9591-a96d6180cd25,Arepas de Queso,https://www.mycolombianrecipes.com/es/arepas-d...,2,1 taza de agua tibia,1,1.000000,NaN,NaN,taza,agua tibia,NaN,NaN,NaN,NaN,NaN,1 taza de agua tibia,agua tibia,taza
2,86af61e4-e16a-11ed-9591-a96d6180cd25,Arepas de Queso,https://www.mycolombianrecipes.com/es/arepas-d...,3,⅓ taza de queso mozzarella o queso blanco rallado,⅓,0.333333,NaN,NaN,taza,queso mozzarella o queso blanco rallado,NaN,NaN,NaN,NaN,NaN,1 3 taza de queso mozzarella o queso blanco ra...,queso mozzarella o queso blanco rallado,taza
3,86af61e4-e16a-11ed-9591-a96d6180cd25,Arepas de Queso,https://www.mycolombianrecipes.com/es/arepas-d...,4,2 cucharadas de mantequilla,2,2.000000,NaN,NaN,cucharadas,mantequilla,NaN,NaN,NaN,NaN,NaN,2 cucharadas de mantequilla,mantequilla,cucharada
4,86af61e4-e16a-11ed-9591-a96d6180cd25,Arepas de Queso,https://www.mycolombianrecipes.com/es/arepas-d...,5,Sal,NaN,NaN,NaN,NaN,NaN,Sal,NaN,NaN,NaN,NaN,NaN,sal,sal,NaN


In [8]:
# ============================================================
# 8. Comparación antes vs después
# ============================================================

recetas_limpias[
    [
        col_linea,
        "ingrediente_linea_limpia",
        col_ingrediente,
        "ingrediente_nombre_limpio",
        col_unidad,
        "unidad_limpia"
    ]
].head(20)

,ingrediente_linea,ingrediente_linea_limpia,ingrediente_nombre,ingrediente_nombre_limpio,unidad,unidad_limpia
0,1 taza de harina de arepa blanca o amarilla,1 taza de harina de arepa blanca o amarilla,harina de arepa blanca o amarilla,harina de arepa blanca o amarilla,taza,taza
1,1 taza de agua tibia,1 taza de agua tibia,agua tibia,agua tibia,taza,taza
2,⅓ taza de queso mozzarella o queso blanco rallado,1 3 taza de queso mozzarella o queso blanco ra...,queso mozzarella o queso blanco rallado,queso mozzarella o queso blanco rallado,taza,taza
3,2 cucharadas de mantequilla,2 cucharadas de mantequilla,mantequilla,mantequilla,cucharadas,cucharada
4,Sal,sal,Sal,sal,NaN,NaN
5,8 muslos de pollo sin la piel,8 muslos de pollo sin la piel,muslos de pollo sin la piel,muslos de pollo sin la piel,NaN,NaN
6,1 cucharada de aceite vegetal,1 cucharada de aceite vegetal,aceite vegetal,aceite vegetal,cucharada,cucharada
7,½ taza de cebolla picada,1 2 taza de cebolla picada,cebolla picada,cebolla picada,taza,taza
8,½ de taza de pimientón rojo picado,1 2 de taza de pimienton rojo picado,de taza de pimientón rojo picado,de taza de pimienton rojo picado,NaN,NaN
9,1 diente de ajo picado,1 diente de ajo picado,ajo picado,ajo picado,diente,diente


In [9]:
# ============================================================
# 9. Análisis de resultados de limpieza
# ============================================================

print("Top ingredientes limpios:")
recetas_limpias["ingrediente_nombre_limpio"].value_counts().head(20)

Top ingredientes limpios:


,count
ingrediente_nombre_limpio,
agua,109
azucar,102
comino molido,91
sal,78
sal y pimienta al gusto,69
sal y pimienta,64
mantequilla,60
ajo picados,47
extracto de vainilla,46


In [10]:
# ============================================================
# 10. Validación de limpieza
# ============================================================

recetas_limpias[
    ["ingrediente_linea", "ingrediente_linea_limpia"]
].sample(15, random_state=42)

,ingrediente_linea,ingrediente_linea_limpia
16683,4 chuletas de ternera de 200 gr. cada una 30 g...,4 chuletas de ternera de 200 gr. cada una 30 g...
4148,1 libra de lomo de res cortado en tajadas delg...,1 libra de lomo de res cortado en tajadas delg...
2527,1 aguacate pelado y en rodajas,1 aguacate pelado y en rodajas
17470,200 gramos de carne picada de ternera 1 pizca ...,200 gramos de carne picada de ternera 1 pizca ...
2388,4 cucharaditas de azúcar,4 cucharaditas de azucar
10397,Agua c/n1 y 1/2 kg. de camarón mediano4 cdas. ...,agua c/n1 y 1/2 kg. de camaron mediano4 cdas. ...
16384,Una cucharada de mantequilla Medio litro de na...,una cucharada de mantequilla medio litro de na...
28965,1 kilo de auyama 8 tazas de agua 3 cucharadas ...,1 kilo de auyama 8 tazas de agua 3 cucharadas ...
5623,500 gr. de arroz carnaroli 1/2 lt. de leche 6 ...,500 gr. de arroz carnaroli 1/2 lt. de leche 6 ...
1723,1 cucharada de aceite de oliva,1 cucharada de aceite de oliva


In [11]:
# ============================================================
# 11. Guardado del dataset limpio
# ============================================================

import os

os.makedirs("data/interim", exist_ok=True)

recetas_limpias.to_csv("data/interim/recetas_limpias.csv", index=False)

print("Dataset limpio guardado correctamente")

Dataset limpio guardado correctamente


In [12]:
# ============================================================
# 12. Conclusiones
# ============================================================

print("""
CONCLUSIONES DE LIMPIEZA Y NORMALIZACIÓN

1. Se realizó la normalización textual del dataset de recetas, incluyendo:
   - conversión a minúsculas,
   - eliminación de tildes,
   - eliminación de caracteres especiales,
   - normalización de espacios.

2. Se generaron nuevas variables:
   - ingrediente_linea_limpia
   - ingrediente_nombre_limpio
   - unidad_limpia

3. La columna ingrediente_linea_limpia será utilizada como entrada para el procesamiento NLP.

4. La columna ingrediente_nombre_limpio será utilizada para procesos de estandarización y matching con la TCAC.

5. Se realizó una primera normalización de unidades, reduciendo la variabilidad léxica.

6. Este proceso constituye la base para las siguientes etapas del pipeline, particularmente tokenización, lematización y extracción de entidades.
""")


CONCLUSIONES DE LIMPIEZA Y NORMALIZACIÓN

1. Se realizó la normalización textual del dataset de recetas, incluyendo:
   - conversión a minúsculas,
   - eliminación de tildes,
   - eliminación de caracteres especiales,
   - normalización de espacios.

2. Se generaron nuevas variables:
   - ingrediente_linea_limpia
   - ingrediente_nombre_limpio
   - unidad_limpia

3. La columna ingrediente_linea_limpia será utilizada como entrada para el procesamiento NLP.

4. La columna ingrediente_nombre_limpio será utilizada para procesos de estandarización y matching con la TCAC.

5. Se realizó una primera normalización de unidades, reduciendo la variabilidad léxica.

6. Este proceso constituye la base para las siguientes etapas del pipeline, particularmente tokenización, lematización y extracción de entidades.

